# 第3课：Residual Bootstrap与2026产量情景

这是课程作业Monte Carlo模型的第一段随机模拟。本课只生成10,000个2026产量情景，不加入期货价格、basis、套保合约或利润。

请从上到下逐个运行代码单元格。

## 0. 为什么不能直接随机抽历史产量？

如果直接从1996–2025年的实际产量中抽样，就会把1990年代较低的技术水平当成2026年的正常水平。

因此我们把产量拆成：

$$FinalYield_i = JulyYieldForecast_i + ResidualDraw_i$$

其中：

$$JulyYieldForecast_i = \beta_0 + \beta_1(2026-1996)+\beta_2PDSIDraw_i$$

这样既保留2026年的长期技术趋势，又保留历史中模型无法解释的产量波动。

## 1. 导入工具并锁定模拟设置

`random seed`让随机结果可以重复。它不代表这个seed更可能发生，只是方便老师和团队复核。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

START_YEAR = 1996
FORECAST_YEAR = 2026
N_SIMULATIONS = 10_000
RANDOM_SEED = 8_122_026

print('模拟次数:', N_SIMULATIONS)
print('Random seed:', RANDOM_SEED)

## 2. 建立已经核对的历史产量与July PDSI数据

产量来自USDA NASS，July PDSI来自NOAA NCEI。教学版将30年数据内置在Notebook中。

In [ ]:
data = pd.DataFrame({
    'year': list(range(1996, 2026)),
    'yield_bu_per_acre': [
        138.0, 138.0, 145.0, 149.0, 144.0, 146.0, 163.0, 157.0,
        181.0, 173.0, 166.0, 171.0, 171.0, 181.0, 165.0, 172.0,
        137.0, 164.0, 178.0, 192.0, 203.0, 202.0, 196.0, 198.0,
        177.0, 204.0, 200.0, 201.0, 211.0, 210.0
    ],
    'july_pdsi': [
        1.51, -0.36, 2.39, 3.10, 1.11, -0.37, 0.06, 0.78,
        1.47, -0.21, -2.39, 1.27, 6.45, 3.98, 6.69, -0.23,
        -3.56, -0.68, 2.27, 3.18, 3.50, 2.06, 1.53, 4.70,
        -0.36, -1.43, -0.89, -1.99, 2.43, 2.54
    ],
})

assert len(data) == 30
assert not data.isna().any().any()
assert not data['year'].duplicated().any()
data['trend_index'] = data['year'] - START_YEAR

data.head()

## 3. 重新估计第2课的Trend + PDSI模型

为了让本Notebook可以独立运行，我们先重建第2课的回归方程。

In [ ]:
X = np.column_stack([
    np.ones(len(data)),
    data['trend_index'].to_numpy(dtype=float),
    data['july_pdsi'].to_numpy(dtype=float),
])
y = data['yield_bu_per_acre'].to_numpy(dtype=float)

beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
intercept, trend_beta, pdsi_beta = beta

data['fitted_yield'] = X @ beta
data['yield_residual'] = data['yield_bu_per_acre'] - data['fitted_yield']

print(
    f'Yield = {intercept:.4f} + {trend_beta:.4f} × Trend Index '    f'+ {pdsi_beta:.4f} × July PDSI + Residual'
)

## 4. 检查Residual库

Residual是模型没有解释的产量差异：

$$Residual_t=ActualYield_t-FittedYield_t$$

因为回归包含截距，Residual平均值应该非常接近0。

In [ ]:
residual_library = data[['year', 'yield_residual']].copy()

residual_summary = pd.Series({
    'count': len(residual_library),
    'mean': residual_library['yield_residual'].mean(),
    'sample_standard_deviation': residual_library['yield_residual'].std(ddof=1),
    'minimum': residual_library['yield_residual'].min(),
    'maximum': residual_library['yield_residual'].max(),
})

assert abs(residual_summary['mean']) < 1e-10
residual_summary.round(4)

## 5. Bootstrap是什么意思？

Bootstrap就是从这30个历史Residual中**有放回抽样**：

- 每次抽取后把该Residual放回；
- 同一个历史Residual可以被抽中多次；
- 某些Residual可能一次都没有被抽中；
- 不强行假设Residual一定服从正态分布。

下面先只抽10次，观察它的工作方式。

In [ ]:
demo_rng = np.random.default_rng(RANDOM_SEED)
demo_indices = demo_rng.integers(0, len(residual_library), size=10)

demo_draws = residual_library.iloc[demo_indices].reset_index(drop=True)
demo_draws.index = np.arange(1, 11)
demo_draws.index.name = 'draw_number'
demo_draws.round(4)

## 6. 为什么天气年份和Residual年份分开抽？

回归已经使用July PDSI解释天气影响。Residual代表PDSI和趋势没有解释的其他冲击。

在包含截距的OLS样本中，Residual与已经进入模型的Trend和PDSI正交。因此本项目采用一个明确的简化假设：每个模拟情景独立抽取天气年份和Residual年份。

这是建模假设，不是声称现实中的所有冲击都完全独立。每个情景会抽取：

1. 一个历史年份的July PDSI，作为2026天气信号；
2. 一个历史年份的Residual，作为其他产量冲击。

我们保留两个source year，方便老师审计每个随机数来自哪里。

## 7. 生成10,000个2026产量情景

每个情景按以下公式生成：

$$JulyForecast_i=\beta_0+\beta_1(30)+\beta_2PDSIDraw_i$$

$$FinalYield_i=\max(0,JulyForecast_i+ResidualDraw_i)$$

`max(0, ...)`是安全检查，避免产量出现不可能的负数。

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

weather_indices = rng.integers(0, len(data), size=N_SIMULATIONS)
residual_indices = rng.integers(0, len(data), size=N_SIMULATIONS)

weather_source_year = data['year'].to_numpy()[weather_indices]
pdsi_draw = data['july_pdsi'].to_numpy()[weather_indices]
residual_source_year = data['year'].to_numpy()[residual_indices]
residual_draw = data['yield_residual'].to_numpy()[residual_indices]

forecast_trend_index = FORECAST_YEAR - START_YEAR
july_yield_forecast = (
    intercept
    + trend_beta * forecast_trend_index
    + pdsi_beta * pdsi_draw
)
final_yield = np.maximum(0.0, july_yield_forecast + residual_draw)

yield_scenarios = pd.DataFrame({
    'scenario_id': np.arange(1, N_SIMULATIONS + 1),
    'weather_source_year': weather_source_year,
    'july_pdsi': pdsi_draw,
    'residual_source_year': residual_source_year,
    'yield_residual_bu_per_acre': residual_draw,
    'july_yield_forecast_bu_per_acre': july_yield_forecast,
    'final_yield_bu_per_acre': final_yield,
})

yield_scenarios.head(10).round(2)

## 8. 验证随机模拟可以重复

使用相同seed重新生成索引，应该得到完全相同的结果。

In [ ]:
check_rng = np.random.default_rng(RANDOM_SEED)
check_weather_indices = check_rng.integers(0, len(data), size=N_SIMULATIONS)
check_residual_indices = check_rng.integers(0, len(data), size=N_SIMULATIONS)

assert np.array_equal(weather_indices, check_weather_indices)
assert np.array_equal(residual_indices, check_residual_indices)
assert len(yield_scenarios) == N_SIMULATIONS
assert yield_scenarios.isna().sum().sum() == 0
assert (yield_scenarios['final_yield_bu_per_acre'] >= 0).all()
assert set(yield_scenarios['weather_source_year']).issubset(set(data['year']))
assert set(yield_scenarios['residual_source_year']).issubset(set(data['year']))

print('检查通过：结果可重复、无缺失、无负产量、source year有效。')

## 9. 总结模拟产量分布

Monte Carlo不只报告一个平均值，还要展示分布和尾部结果。

In [ ]:
yield_distribution_summary = pd.Series({
    'n_scenarios': len(yield_scenarios),
    'mean_final_yield': yield_scenarios['final_yield_bu_per_acre'].mean(),
    'standard_deviation': yield_scenarios['final_yield_bu_per_acre'].std(ddof=1),
    'minimum': yield_scenarios['final_yield_bu_per_acre'].min(),
    '5th_percentile': yield_scenarios['final_yield_bu_per_acre'].quantile(0.05),
    'median': yield_scenarios['final_yield_bu_per_acre'].median(),
    '95th_percentile': yield_scenarios['final_yield_bu_per_acre'].quantile(0.95),
    'maximum': yield_scenarios['final_yield_bu_per_acre'].max(),
})

yield_distribution_summary.round(2)

### 怎样解释这些结果？

- Mean是10,000个情景的平均产量，不是保证值；
- Standard deviation表示产量风险；
- 5th percentile表示大约5%的模拟产量低于该值；
- 这些情景以后会与价格、basis和套保策略结合。

## 10. 保存本课作业结果

In [ ]:
output_dir = Path.cwd() / 'lesson_output' / 'step_03'
output_dir.mkdir(parents=True, exist_ok=True)

residual_library.to_csv(output_dir / 'historical_yield_residual_library.csv', index=False)
yield_scenarios.to_csv(output_dir / 'yield_scenarios_10000.csv', index=False)

summary = {
    'n_simulations': N_SIMULATIONS,
    'random_seed': RANDOM_SEED,
    'simulation_formula': (
        'FinalYield = max(0, Intercept + TrendBeta*30 + PDSIBeta*PDSIDraw '
        '+ ResidualDraw)'
    ),
    'mean_final_yield_bu_per_acre': float(yield_distribution_summary['mean_final_yield']),
    'standard_deviation_bu_per_acre': float(yield_distribution_summary['standard_deviation']),
    'fifth_percentile_bu_per_acre': float(yield_distribution_summary['5th_percentile']),
    'median_bu_per_acre': float(yield_distribution_summary['median']),
    'ninety_fifth_percentile_bu_per_acre': float(yield_distribution_summary['95th_percentile']),
    'important_limit': 'This lesson simulates yield only; price and hedging are not included.',
}

with (output_dir / 'step_03_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)

print('第3课结果已保存到：', output_dir)

## 本课完成标准

固定seed下，运行正确时应该得到大约：

- 10,000个产量情景；
- Mean final yield：**210.33 bu/acre**；
- Standard deviation：**11.34 bu/acre**；
- 5th percentile：**189.90 bu/acre**；
- Median：**210.94 bu/acre**；
- 95th percentile：**227.91 bu/acre**。

到这里停止。下一课才会建立July和harvest December futures price模型。